<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:10px solid #2a78d6;border-radius:8px;padding:26px 30px;margin:6px 0 20px 0;"><div style="color:#c04b1c;font-size:12px;font-weight:700;letter-spacing:.16em;text-transform:uppercase;margin-bottom:10px;">Exploratory data analysis</div><h1 style="color:#0b0b0b;font-size:31px;line-height:1.22;margin:0;font-weight:700;letter-spacing:-.01em;">RSNA Knee Abnormality Detection: a first look</h1><div style="color:#6a6862;font-size:13px;margin-top:12px;">Twelve findings per study &middot; macro-averaged ROC AUC &middot; 4,407 training studies, 58 labelled &middot; 265 GB</div></div>

Twelve findings per knee MRI study, scored as the macro average of twelve ROC AUCs.
4,407 training studies, 24,371 series, roughly 730,000 slices, 265 GB.

Two facts reshape the whole problem, and both are visible in the first ten minutes of
looking at the CSVs.

**58 of the 4,407 training studies carry labels.** That is 1.3%. The other 4,349 studies
ship a free-text radiology report and nothing else. The competition is a label-extraction
problem wearing an imaging-competition costume, and the 58 gold studies are far too small
and far too positive-enriched to rank two models against each other. Section 2 puts a
number on that: a bootstrap 95% interval for macro AUC on the gold 58 spans about 0.079,
which is an order of magnitude wider than the margins that separate leaderboard positions.

**The reports exist at training time only.** `test.csv` has one column,
`StudyInstanceUID`. There is no report at inference. Text is a label source and an
auxiliary training signal; a pipeline that reads report text to predict scores nothing.
Section 4 puts the two schemas one above the other.

Everything below is computed from the competition data at run time. Nothing is
hard-coded from a previous run. The notebook runs against the Kaggle mount and against a
partial local mirror, and every DICOM walk is bounded by a constant declared in the config
cell at the top of section 0.

> **How this was made.** This notebook was written automatically by Claude Code running a
> custom multi-agent LLM harness: parallel agents drafted the analysis, then separate
> adversarial verifier agents executed every cell, checked every printed number, opened
> every figure, and sent defects back to the builder script. It is published to Kaggle with
> the Kaggle CLI, which is also how the companion baseline notebook is submitted. Full
> detail is in the last section. **Code cells are collapsed by default** so this reads as a
> report; click any *Show code* toggle to expand one.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">0</span><span style="vertical-align:middle;">Setup: input root, sampling budget, design system</span></h2></div>

Three cells: find the data, declare every sampling bound in one place, then define the
visual system once so the rest of the notebook inherits it.

In [ ]:
"""Locate the competition data and declare every sampling bound."""

import os
import re
import sys
import math
import json
import warnings
import collections
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- Sampling budget -------------------------------------------------------
# Every DICOM walk in this notebook is bounded by one of these. Nothing here
# iterates all ~730k slices; a Kaggle CPU session would not finish.
SEED = 20260805                # every random draw in this notebook uses it
N_HEADER_STUDIES = 90          # studies sampled for the DICOM header census
MAX_HEADER_SERIES = 420        # hard cap on header reads (stop_before_pixels)
MAX_FALLBACK_FILES = 400       # cap when the nested layout is absent (local runs)
N_MONTAGE_SLICES = 4           # slices shown per plane in the pixel montage
N_MONTAGE_CANDIDATES = 8       # series tried per plane before giving up
N_INTENSITY_SERIES = 6         # series in the intensity-distribution panel
N_BOOTSTRAP = 2000             # bootstrap resamples for the AUC noise floor

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion",
    "Fracture",
]

RNG = np.random.default_rng(SEED)


def find_root():
    """Locate the directory holding ``train.csv``, Kaggle mount first.

    Returns:
        A ``Path`` to the data root.

    Raises:
        FileNotFoundError: If no candidate directory contains ``train.csv``.
    """
    here = Path.cwd()
    candidates = [
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    ]
    for up in [here, *here.parents][:5]:
        candidates.append(up / "data" / "kaggle_mirror" / "rsna-knee-abnormality-detection")
        candidates.append(up / "data" / "raw")
    for c in candidates:
        if (c / "train.csv").is_file():
            return c
    # Last resort: a shallow scan of /kaggle/input.
    base = Path("/kaggle/input")
    if base.is_dir():
        for child in sorted(base.iterdir()):
            if (child / "train.csv").is_file():
                return child
    raise FileNotFoundError(
        "Could not locate train.csv. Checked: " + ", ".join(str(c) for c in candidates)
    )


ROOT = find_root()
ON_KAGGLE = str(ROOT).startswith("/kaggle/")

# Where DICOMs live, if they are here at all.
TRAIN_DCM_DIR = ROOT / "train_series"
TEST_DCM_DIR = ROOT / "test_series"

# A local checkout keeps a handful of real DICOMs outside the mirror so this
# notebook can be exercised offline. Harmless and absent on Kaggle.
EXTRA_DCM_DIRS = [p for p in
                  [ROOT.parent / "sample", ROOT.parent.parent / "data" / "sample"]
                  if p.is_dir()]

print("data root      :", ROOT)
print("running on     :", "Kaggle" if ON_KAGGLE else "local mirror")
print("train DICOM dir:", TRAIN_DCM_DIR, "->", "present" if TRAIN_DCM_DIR.is_dir() else "ABSENT")
print("test DICOM dir :", TEST_DCM_DIR, "->", "present" if TEST_DCM_DIR.is_dir() else "ABSENT")
if EXTRA_DCM_DIRS:
    print("extra DICOM dir:", ", ".join(str(p) for p in EXTRA_DCM_DIRS))
print("csv files      :", sorted(p.name for p in ROOT.glob("*.csv")))

In [ ]:
"""Define the design system once. Every later chart inherits it."""

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe

# ---- Ink and surface -------------------------------------------------------
# Three text tiers, every one of them at or above the WCAG AA 4.5:1 floor
# against SURFACE. Ratios are computed, not eyeballed: 19.2, 7.7, 5.4. The two
# structural greys below carry no text and are deliberately left light, because
# darkening a gridline adds noise and contrast rules do not apply to it.
SURFACE = "#fcfcfb"   # chart surface
INK = "#0b0b0b"       # primary text                    19.2:1
INK_2 = "#52514e"     # secondary text, value annotations 7.7:1
MUTED = "#6a6862"     # axis labels, tick labels, captions 5.4:1
GRID = "#e1e0d9"      # hairline gridlines, non-text
AXIS = "#c3c2b7"      # baseline, non-text

# ---- Categorical palette ---------------------------------------------------
# Five hues, one per anatomical family. Chosen by exhaustive search over the
# eight-slot source palette and validated all-pairs: worst pair CVD dE 13.0,
# worst pair normal-vision dE 16.3 (OKLab x100). Twelve separate hues would
# fail every colourblind-safety check, so colour carries the FAMILY and the
# finding name is always written on the axis. No information is colour-alone.
FAMILY_COLORS = {
    "Ligament": "#2a78d6",            # blue
    "Meniscus": "#eda100",            # yellow
    "Osteoarthritis": "#e87ba4",      # magenta
    "Effusion & synovium": "#008300",  # green
    "Bone trauma": "#4a3aa7",         # violet
}
LABEL_FAMILY = {
    "ACL": "Ligament", "MCL": "Ligament",
    "Medial Meniscus": "Meniscus", "Lateral Meniscus": "Meniscus",
    "Medial OA": "Osteoarthritis", "Lateral OA": "Osteoarthritis",
    "PF OA": "Osteoarthritis",
    "Effusion": "Effusion & synovium", "Synovitis": "Effusion & synovium",
    "Baker's": "Effusion & synovium",
    "Contusion": "Bone trauma", "Fracture": "Bone trauma",
}
LABEL_COLORS = {k: FAMILY_COLORS[v] for k, v in LABEL_FAMILY.items()}

PRIMARY = "#2a78d6"    # the single-series colour, marks only
ACCENT = "#eb6834"     # emphasis: the one thing the chart is about, marks only
NEUTRAL = "#c3c2b7"    # de-emphasised bars

# Text-safe partners for the two mark colours. A mark does not have to clear a
# contrast floor; the sentence next to it does. ACCENT at 3.1:1 is fine as a
# bar and unreadable as a caption, so annotations that belong to an accent mark
# are drawn in ACCENT_TEXT (4.8:1) and fills that carry SURFACE-coloured text
# use PRIMARY_DEEP (5.3:1) rather than PRIMARY (4.3:1).
ACCENT_TEXT = "#c04b1c"   # accent annotations           4.8:1 on SURFACE
PRIMARY_DEEP = "#256abf"  # fill behind SURFACE text     5.3:1 either way

# ---- Sequential ramp (one hue, light to dark) ------------------------------
BLUE_STEPS = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
              "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281",
              "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("knee_blue", BLUE_STEPS)


# ---- Contrast ---------------------------------------------------------------
def _rel_lum(colour):
    """WCAG relative luminance of a colour.

    Args:
        colour: Anything matplotlib can parse as a colour.

    Returns:
        Relative luminance in ``[0, 1]``.
    """
    total = 0.0
    for weight, chan in zip((0.2126, 0.7152, 0.0722), mpl.colors.to_rgb(colour)):
        lin = chan / 12.92 if chan <= 0.03928 else ((chan + 0.055) / 1.055) ** 2.4
        total += weight * lin
    return total


def contrast(fg, bg):
    """WCAG 2.x contrast ratio between two colours.

    Args:
        fg: Foreground colour.
        bg: Background colour.

    Returns:
        A ratio between 1.0 and 21.0. AA wants 4.5 for normal-size text.
    """
    lo, hi = sorted((_rel_lum(fg), _rel_lum(bg)))
    return (hi + 0.05) / (lo + 0.05)


def ink_on(fill):
    """Pick whichever of INK and SURFACE reads better on a filled mark.

    A number written inside a bar or a heatmap cell sits on the mark, not on
    the page, so the page's three text tiers do not apply to it. Choosing by
    computed contrast beats a hand-tuned lightness threshold, which is always
    wrong somewhere in the middle of a sequential ramp.

    Args:
        fill: The colour of the mark the text sits on.

    Returns:
        ``INK`` or ``SURFACE``.
    """
    return INK if contrast(INK, fill) >= contrast(SURFACE, fill) else SURFACE


def _seq_safe(t):
    """Sample the sequential ramp, stepping over its unreadable stretch.

    About 58% of the way up the ramp the blue is mid-toned enough that neither
    INK nor SURFACE clears 4.5:1 on it, so a number written in a cell of that
    colour cannot be made readable by choosing the text colour. The band is 4%
    of the ramp wide; nudging a value to the nearer edge is invisible in the
    shading and makes every annotated cell legible.

    Args:
        t: Position on the ramp in ``[0, 1]``.

    Returns:
        An RGBA tuple.
    """
    lo, hi = 0.560, 0.620
    if lo < t < hi:
        t = lo if t < (lo + hi) / 2 else hi
    return SEQ(t)


# The ramp actually used for filled cells that carry a number.
SEQ_SAFE = mpl.colors.ListedColormap(
    [_seq_safe(i / 255.0) for i in range(256)], name="knee_blue_safe")


mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "figure.dpi": 118,
    "savefig.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelcolor": MUTED,
    "axes.labelsize": 12.5,
    "axes.titlesize": 13.5,
    "axes.grid": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "legend.frameon": False,
    "legend.fontsize": 12,
    "lines.linewidth": 2.4,
    "lines.markersize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Segoe UI", "Helvetica", "Arial"],
    "font.size": 12.5,
    "text.color": INK,
})


def style(ax, grid_axis="x"):
    """Apply the recessive grid and spine treatment to one axes.

    Args:
        ax: A matplotlib axes.
        grid_axis: ``"x"``, ``"y"`` or ``"none"``. Gridlines run along this axis
            only, behind the marks.

    Returns:
        The same axes, for chaining.
    """
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.grid(False)
    if grid_axis == "x":
        ax.xaxis.grid(True, zorder=0)
    elif grid_axis == "y":
        ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    return ax


def _wrap(text, width_in, fontsize):
    """Wrap a header or caption string to the figure's own width.

    The notebook's inline backend saves with a tight bounding box, so a text
    run wider than the figure silently widens the exported PNG and leaves the
    plot stranded in one corner. Wrapping to the figure width prevents that.

    Args:
        text: The string to wrap.
        width_in: Figure width in inches.
        fontsize: Point size the string will be drawn at.

    Returns:
        The string with newlines inserted.
    """
    import textwrap
    chars = max(24, int((width_in - 0.15) * 72.0 / (fontsize * 0.53)))
    return "\n".join(textwrap.wrap(text, chars)) if text else ""


def finish(fig, finding, detail="", caption="", tight=True, headroom=0.0,
           footroom=0.0):
    """Add the finding title, optional detail line and caption, then show.

    Titles state the finding, not the mechanism. All reserved space is computed
    in inches from the wrapped line count, so a tall figure and a short one get
    the same visual header band.

    Args:
        fig: The figure to close out.
        finding: The headline. One sentence, stating what the chart shows.
        detail: Optional second line in secondary ink.
        caption: Optional footer in muted ink, for provenance and caveats.
        tight: Run ``tight_layout``. Set ``False`` for image grids that manage
            their own spacing.
        headroom: Extra inches reserved above the axes, for figures whose
            panels carry their own titles.
        footroom: Extra inches reserved below the axes, on top of the measured
            clearance, for anything the tight bounding box does not see.
    """
    w, h = fig.get_size_inches()
    finding = _wrap(finding, w, 17.5)
    detail = _wrap(detail, w, 13.0)
    caption = _wrap(caption, w, 11.25)
    n_find = finding.count("\n") + 1
    n_det = (detail.count("\n") + 1) if detail else 0
    n_cap = (caption.count("\n") + 1) if caption else 0

    head_in = 0.32 + 0.31 * n_find + (0.10 + 0.25 * n_det) + headroom
    foot_in = ((0.18 + 0.21 * n_cap) if caption else 0.13) + footroom

    # tight_layout silently does nothing on two whole classes of figure and only
    # emits a UserWarning: any figure holding a fixed-aspect axes (imshow,
    # set_aspect("equal")), and any figure whose gridspec was given an explicit
    # hspace or wspace, which is every stacked pair in this notebook. So the
    # header band cannot be left to it. Run it for the tick-label margins, then
    # clamp top and bottom directly, which always applies.
    if tight:
        try:
            fig.tight_layout(rect=(0.0, foot_in / h, 1.0, 1.0 - head_in / h))
        except Exception:
            pass
    sp = fig.subplotpars
    top = min(sp.top, 1.0 - head_in / h)
    bottom = max(sp.bottom, foot_in / h)
    if top > bottom + 0.05:
        fig.subplots_adjust(top=top, bottom=bottom)

    # When tight_layout declined, the axes box moved but its tick labels and
    # axis label still hang below it, unmeasured, and the caption is pinned to
    # the figure edge. Measure the real overhang and lift the axes if the two
    # would meet. Only ever lifts, never lowers, and never past a sane cap.
    if caption:
        try:
            fig.canvas.draw()
            rend = fig.canvas.get_renderer()
            low = min(a.get_tightbbox(rend).y0 for a in fig.axes) / fig.bbox.height
            shift = min(foot_in / h - low, 0.25)
            sp = fig.subplotpars
            if shift > 0.002 and sp.top > sp.bottom + shift + 0.05:
                fig.subplots_adjust(bottom=sp.bottom + shift)
        except Exception:
            pass

    fig.text(0.008, 1.0 - 0.20 / h, finding, ha="left", va="top",
             fontsize=17.5, fontweight="bold", color=INK, linespacing=1.25)
    if detail:
        fig.text(0.008, 1.0 - (0.25 + 0.31 * n_find) / h, detail, ha="left",
                 va="top", fontsize=13, color=INK_2, linespacing=1.3)
    if caption:
        fig.text(0.008, 0.09 / h, caption, ha="left", va="bottom",
                 fontsize=11.25, color=MUTED, linespacing=1.3)
    plt.show()


def hbar(ax, names, values, colors=None, fmt="{:,.0f}", annot=None,
         pad_frac=0.022, xpad=1.18):
    """Draw a horizontal bar chart with every bar directly labelled.

    Horizontal bars sorted by value beat vertical bars with rotated ticks, and a
    value written at the bar end beats making the reader trace back to an axis.
    Annotations wear secondary ink, never the series colour.

    Args:
        ax: Target axes.
        names: Category names, drawn top to bottom in the order given.
        values: Numeric values, same length as ``names``.
        colors: Per-bar colours, or ``None`` for the single-series colour.
        fmt: Format string applied to each value when ``annot`` is ``None``.
        annot: Pre-formatted annotation strings, one per bar, overriding ``fmt``.
        pad_frac: Annotation offset as a fraction of the axis range.
        xpad: Right-hand headroom multiplier, so annotations never clip.

    Returns:
        The bar container.
    """
    names = list(names)
    values = np.asarray(values, dtype=float)
    y = np.arange(len(names))[::-1]
    if colors is None:
        colors = [PRIMARY] * len(names)
    if annot is None:
        annot = [fmt.format(v) for v in values]
    bars = ax.barh(y, values, height=0.70, color=colors, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=12.5, color=INK)
    top = float(values.max()) if len(values) else 1.0
    pad = top * pad_frac
    for yy, v, txt in zip(y, values, annot):
        ax.text(v + pad, yy, txt, va="center", ha="left", fontsize=12,
                color=INK_2)
    ax.set_xlim(0, top * xpad)
    ax.set_ylim(-0.75, len(names) - 0.25)
    style(ax, "x")
    ax.spines["left"].set_visible(False)
    return bars


def family_legend(ax, loc="lower right", ncol=1):
    """Attach the fixed anatomical-family colour legend to an axes.

    Args:
        ax: Target axes.
        loc: Matplotlib legend location.
        ncol: Number of legend columns.

    Returns:
        The legend object.
    """
    handles = [Line2D([0], [0], marker="s", linestyle="none", markersize=11,
                      markerfacecolor=c, markeredgecolor="none", label=k)
               for k, c in FAMILY_COLORS.items()]
    return ax.legend(handles=handles, loc=loc, ncol=ncol, handletextpad=0.5,
                     labelcolor=INK_2, borderaxespad=0.6)


def show_table(df, caption=""):
    """Render a small dataframe as the table view that accompanies a chart.

    Args:
        df: The dataframe to display.
        caption: Optional caption printed beneath.
    """
    from IPython.display import display
    display(df)
    if caption:
        print(caption)


print("design system ready:", len(FAMILY_COLORS), "family hues,",
      len(BLUE_STEPS), "sequential steps")
print("text contrast on SURFACE (WCAG AA floor 4.5):",
      ", ".join(f"{n} {contrast(c, SURFACE):.1f}" for n, c in
                [("INK", INK), ("INK_2", INK_2), ("MUTED", MUTED),
                 ("ACCENT_TEXT", ACCENT_TEXT)]))

In [ ]:
"""Load the five competition CSVs."""

train = pd.read_csv(ROOT / "train.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test = pd.read_csv(ROOT / "test.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

for name, df in [("train", train), ("train_series", train_series),
                 ("test", test), ("test_series", test_series),
                 ("sample_submission", sample_sub)]:
    print(f"{name:18s} {df.shape[0]:>7,} x {df.shape[1]:<3d}  {list(df.columns)[:4]}")

# The submission column order is fixed by sample_submission.csv. Check that the
# label list used throughout this notebook matches it exactly.
assert list(sample_sub.columns) == ["StudyInstanceUID"] + LABELS, \
    "submission column order does not match the LABELS constant"
print("\nsubmission column order matches LABELS: OK")

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">1</span><span style="vertical-align:middle;">The shape of the problem</span></h2></div>

One study is one prediction row of twelve probabilities. Underneath each study sit five or
six series, and underneath each series sit twenty to forty-five slices, with a tail that
runs into the hundreds. The scored unit is therefore three levels above the unit the model
actually sees, and something has to aggregate slices to series to study.

In [ ]:
"""Headline inventory as stat tiles. The number is the chart."""

n_studies = train["StudyInstanceUID"].nunique()
n_series = len(train_series)
n_test_series = len(test_series)

tiles = [
    (f"{n_studies:,}", "training studies", "one prediction row each"),
    (f"{n_series:,}", "training series", f"{n_series / n_studies:.2f} per study"),
    ("12", "findings per study", "each scored separately"),
    ("macro AUC", "the metric", "a rare label costs as much as a common one"),
]

# One axes, four cards drawn on it. These are stat tiles, not analysis panels:
# no axis, no marks, nothing to compare along a shared scale. Drawing them as
# rectangles on a single axes rather than as a 2 x 2 grid of subplots keeps the
# notebook's one-or-two-panel rule literally true, and a 2 x 2 block of cards
# reads better than four thin cards across a wide column.
fig, ax = plt.subplots(figsize=(13.5, 5.4))
ax.axis("off")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
# The gutters are asymmetric in axes fraction because the axes is about three
# times wider than it is tall; both work out to roughly a fifth of an inch.
CARD_W, GAP_X = 0.492, 0.016
GAP_Y = 0.055
CARD_H = (1.0 - GAP_Y) / 2.0
for k, (big, mid, small) in enumerate(tiles):
    x0 = 0.0 if k % 2 == 0 else CARD_W + GAP_X
    y0 = CARD_H + GAP_Y if k < 2 else 0.0
    ax.add_patch(plt.Rectangle((x0, y0), CARD_W, CARD_H, facecolor=SURFACE,
                               edgecolor=GRID, linewidth=1.2, zorder=0))
    ax.add_patch(plt.Rectangle((x0, y0), 0.011, CARD_H, facecolor=PRIMARY,
                               edgecolor="none", zorder=1))
    size = 38 if len(big) <= 6 else 25
    tx = x0 + 0.030
    ax.text(tx, y0 + CARD_H * 0.70, big, fontsize=size, fontweight="bold",
            color=INK, va="center", ha="left")
    ax.text(tx, y0 + CARD_H * 0.37, mid, fontsize=14, color=INK_2,
            va="center", ha="left")
    ax.text(tx, y0 + CARD_H * 0.17, small, fontsize=12, color=MUTED,
            va="center", ha="left")

finish(fig, "4,407 studies, 24,371 series, 12 findings, one macro-averaged AUC",
       detail="The scored unit is the study. The unit a network sees is the slice, "
              "three levels down.",
       caption=f"Counts read from train.csv and train_series.csv at run time. "
               f"test_series.csv ships {n_test_series} placeholder rows.")

**Finding.** Macro averaging is the design decision that shapes effort allocation. A label
with nine positives in the gold set is worth exactly as much of the final score as
Effusion, which has thirty-five. A model that is near-random on one finding gives up a
twelfth of the achievable margin above 0.5, which is 0.042 of the final score, no matter
how good it is on the other eleven. Rare findings deserve disproportionate attention, not
less.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">2</span><span style="vertical-align:middle;">The label problem</span></h2></div>

This is the section that decides how the competition is won or lost.

In [ ]:
"""How many studies actually carry labels?"""

label_block = train[LABELS]
n_labelled_per_row = label_block.notna().sum(axis=1)
gold = train.loc[n_labelled_per_row == 12].reset_index(drop=True)
unlabelled = train.loc[n_labelled_per_row == 0]

print("label completeness by row")
print(n_labelled_per_row.value_counts().rename("studies").to_string())
print()
print(f"fully labelled studies : {len(gold):,}  ({100 * len(gold) / len(train):.2f}%)")
print(f"unlabelled studies     : {len(unlabelled):,}")
print(f"partially labelled     : {len(train) - len(gold) - len(unlabelled):,}")
print()
print("distinct values in the labelled block:",
      sorted(pd.unique(gold[LABELS].to_numpy().ravel())))

# Every label column is non-null on exactly the same rows?
nn_sets = {c: frozenset(train.index[train[c].notna()]) for c in LABELS}
same = len(set(nn_sets.values())) == 1
print("all twelve label columns share one non-null row set:", same)

In [ ]:
"""Unit chart: one square per training study, the labelled ones highlighted."""

n_total = len(train)
ncols = 147
nrows = int(np.ceil(n_total / ncols))
grid = np.full(nrows * ncols, np.nan)
grid[:len(gold)] = 1.0          # labelled, grouped for legibility
grid[len(gold):n_total] = 0.0   # unlabelled
grid = grid.reshape(nrows, ncols)

fig, ax = plt.subplots(figsize=(14.0, 6.0))
cmap = mpl.colors.ListedColormap([GRID, ACCENT])
ax.pcolormesh(np.arange(ncols + 1), np.arange(nrows + 1)[::-1], grid,
              cmap=cmap, vmin=0, vmax=1, edgecolors=SURFACE, linewidth=0.45)
ax.set_xlim(-1, ncols + 1)
ax.set_ylim(-5.0, nrows + 4.2)
ax.set_aspect("equal")
ax.axis("off")

ax.annotate(f"{len(gold)} studies carry all twelve labels",
            xy=(len(gold) / 2.0, nrows - 0.1),
            xytext=(len(gold) / 2.0, nrows + 2.6),
            fontsize=14.5, color=ACCENT_TEXT, fontweight="bold", va="bottom",
            ha="center",
            arrowprops=dict(arrowstyle="-", color=ACCENT_TEXT, linewidth=1.4))
ax.text(ncols, -1.4, f"{len(unlabelled):,} studies carry a report and nothing else",
        fontsize=13.5, color=INK_2, va="top", ha="right")

finish(fig, "Only 1.3% of training studies carry labels",
       detail=f"{len(gold)} of {n_total:,} studies have all twelve findings filled in. "
              f"The other {len(unlabelled):,} ship a free-text radiology report and nothing else.",
       caption="One square is one study. The labelled studies are grouped at the top left "
               "for legibility; in the file they are scattered.")

**Finding.** The imaging model is downstream of a label pipeline that does not exist yet.
Training a network on 58 studies is not a serious plan for a twelve-label problem; training
it on 4,407 studies requires turning 4,349 radiology reports into 52,188 binary decisions
first. Label quality caps model quality, and the ceiling is set before a single convolution
runs. Budget accordingly: the extraction step deserves more effort than the backbone
choice.

In [ ]:
"""Per-label positive counts among the 58 gold studies."""

pos = gold[LABELS].sum().astype(int)
order = pos.sort_values(ascending=False).index.tolist()
vals = pos[order].to_numpy()
cols = [LABEL_COLORS[l] for l in order]

fig, ax = plt.subplots(figsize=(14.0, 6.8))
hbar(ax, order, vals, cols,
     annot=[f"{v}   {100 * v / len(gold):.0f}%" for v in vals], xpad=1.30)
ax.set_xlabel(f"positive studies (of {len(gold)})")
family_legend(ax, loc="lower right")

finish(fig, "Every finding has between 9 and 35 positives in the gold set",
       detail="MCL is the scarcest at 9. Effusion is the most common at 35. "
              "Both count equally toward the macro average.",
       caption="Colour marks the anatomical family, not the individual finding; "
               "twelve distinct hues would fail colourblind separation. "
               "Percentages are of the 58 gold studies.")

show_table(pd.DataFrame({
    "positives": pos[order],
    "negatives": len(gold) - pos[order],
    "prevalence": (pos[order] / len(gold)).round(3),
    "family": [LABEL_FAMILY[l] for l in order],
}))

**Finding.** MCL carries nine positives. A single mislabelled study moves its AUC by
several points. Any per-label metric computed here is a coin flip dressed as a decimal, and
the two scarcest labels, MCL and Lateral OA, together account for a sixth of the macro
score.

In [ ]:
"""Is the gold set a random sample of studies? Findings per study, and co-occurrence."""

per_study = gold[LABELS].sum(axis=1)

fig = plt.figure(figsize=(13.5, 11.6))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.30], hspace=0.30)

ax0 = fig.add_subplot(gs[0, 0])
counts = per_study.value_counts().sort_index()
xs = np.arange(0, 13)
ys = np.array([counts.get(float(k), 0) for k in xs])
ax0.bar(xs, ys, width=0.72, color=PRIMARY, zorder=3)
top_y = ys.max()
ax0.set_ylim(0, top_y * 1.40)
ax0.plot([0], [0], marker="o", markersize=9, color=ACCENT, zorder=5,
         markeredgecolor=SURFACE, markeredgewidth=2)
# A vertical leader, drawn explicitly. An annotate() arrow to a left-aligned
# label lands on the far edge of the text box and draws a long diagonal across
# the bars, which reads as a data series. The column above x=0 is empty, so a
# straight rule there crosses nothing.
ax0.plot([0.0, 0.0], [top_y * 0.10, top_y * 1.15], color=ACCENT, linewidth=1.3,
         solid_capstyle="butt", zorder=4)
ax0.text(0.12, top_y * 1.25, "the bin at zero is empty", fontsize=12.5,
         color=ACCENT_TEXT, fontweight="bold", va="center", ha="left")
for x, y in zip(xs, ys):
    if y:
        ax0.text(x, y + top_y * 0.025, str(int(y)), ha="center", va="bottom",
                 fontsize=12, color=INK_2)
ax0.axvline(per_study.mean(), color=ACCENT, linewidth=2.0, zorder=4)
ax0.text(per_study.mean() + 0.2, top_y * 1.05, f"mean {per_study.mean():.2f}",
         color=ACCENT_TEXT, fontsize=12.5, fontweight="bold", va="center")
ax0.set_xticks(xs)
ax0.set_xlabel("positive findings in the study")
ax0.set_ylabel("gold studies")
ax0.set_title("Not one study is negative on all twelve", loc="left",
              fontsize=13.5, color=INK, pad=10)
style(ax0, "y")

ax1 = fig.add_subplot(gs[1, 0])
Y = gold[LABELS].to_numpy()
co = Y.T @ Y
# aspect="auto" rather than the imshow default: stacked under a full-width bar
# chart, a square 12 x 12 matrix would sit in a narrow column with empty page
# either side. Wide cells cost nothing here because every cell is annotated.
top_co = float(co.max()) or 1.0
im = ax1.imshow(co, cmap=SEQ_SAFE, vmin=0, vmax=top_co, aspect="auto")
ax1.set_xticks(range(12))
ax1.set_yticks(range(12))
ax1.set_xticklabels(LABELS, rotation=45, ha="right", fontsize=11, color=INK_2)
ax1.set_yticklabels(LABELS, fontsize=11, color=INK_2)
for i in range(12):
    for j in range(12):
        ax1.text(j, i, int(co[i, j]), ha="center", va="center", fontsize=10.5,
                 color=ink_on(SEQ_SAFE(co[i, j] / top_co)))
for s in ax1.spines.values():
    s.set_visible(False)
ax1.tick_params(length=0)
ax1.set_title("Co-occurrence counts; the diagonal is the label's own prevalence",
              loc="left", fontsize=13.5, color=INK, pad=10)

finish(fig, "The gold 58 are positive-enriched, not a random sample of studies",
       detail=f"Every one carries at least one finding, and the mean study carries "
              f"{per_study.mean():.2f} of 12.",
       headroom=0.38, footroom=1.35,
       caption="Top: positive findings per gold study. Bottom: pairwise co-occurrence "
               "among the 58; every cell is annotated, so the chart is its own table view.")

**Finding.** The bin at zero findings is empty. Selection for the gold set was conditional
on abnormality, so prevalence in these 58 studies tells you nothing about prevalence in the
4,349 unlabelled ones, and nothing about the test set. Two consequences follow immediately.
Do not calibrate probabilities to gold-set prevalence. Do not use the gold set to estimate
how often a finding occurs, only to check whether an extracted label agrees with a
radiologist when the two are compared on the same study.

The competition host issues a matching warning: prevalence is not guaranteed to be the same
across training, public leaderboard, and final evaluation. AUC is insensitive to prevalence
in expectation, so this is mostly a warning about variance and about tuning thresholds.

In [ ]:
"""How precisely can 58 studies measure an AUC? Two independent answers."""

from scipy.stats import norm, rankdata


def hanley_se(auc, n_pos, n_neg):
    """Hanley-McNeil standard error of an empirical ROC AUC.

    Args:
        auc: Assumed true area under the ROC curve.
        n_pos: Number of positive cases.
        n_neg: Number of negative cases.

    Returns:
        The standard error of the empirical AUC estimate.
    """
    q1 = auc / (2.0 - auc)
    q2 = 2.0 * auc * auc / (1.0 + auc)
    num = (auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2)
           + (n_neg - 1) * (q2 - auc ** 2))
    return float(np.sqrt(num / (n_pos * n_neg)))


def fast_auc(y, s):
    """ROC AUC via the Mann-Whitney rank statistic, with tie correction.

    Args:
        y: Binary ground truth, shape ``(n,)``.
        s: Scores, shape ``(n,)``.

    Returns:
        The AUC, or ``float("nan")`` when one class is absent.
    """
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    r = rankdata(s)
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


TRUE_AUC = 0.80  # the reference operating point the intervals are computed at

rows = []
for lab in LABELS:
    p = int(pos[lab])
    n = len(gold) - p
    se = hanley_se(TRUE_AUC, p, n)
    rows.append({"label": lab, "pos": p, "neg": n, "se": se, "half_ci": 1.96 * se})
ci = pd.DataFrame(rows).set_index("label").loc[LABELS]

# Bootstrap the macro average over studies, so label correlation is preserved.
mu = np.sqrt(2.0) * norm.ppf(TRUE_AUC)
boot_rng = np.random.default_rng(SEED)
scores = boot_rng.normal(loc=Y * mu, scale=1.0)
idx = np.arange(len(gold))
macro = []
for _ in range(N_BOOTSTRAP):
    b = boot_rng.choice(idx, size=len(gold), replace=True)
    a = [fast_auc(Y[b, j], scores[b, j]) for j in range(12)]
    if not np.any(np.isnan(a)):
        macro.append(float(np.mean(a)))
macro = np.asarray(macro)
lo, hi = np.percentile(macro, [2.5, 97.5])

print(f"assumed true AUC per label      : {TRUE_AUC:.2f}")
print(f"mean per-label standard error   : {ci['se'].mean():.4f}")
print(f"widest per-label 95% interval   : +/- {ci['half_ci'].max():.3f} ({ci['half_ci'].idxmax()})")
print(f"bootstrap macro AUC 95% interval: [{lo:.4f}, {hi:.4f}]  width {hi - lo:.4f}")
print(f"bootstrap resamples used        : {len(macro):,} of {N_BOOTSTRAP:,}")

In [ ]:
"""Chart the noise floor: per-label intervals, and the macro-average interval."""

fig = plt.figure(figsize=(13.5, 11.2))
gs = fig.add_gridspec(2, 1, height_ratios=[1.35, 1.0], hspace=0.32)

ax0 = fig.add_subplot(gs[0, 0])
o = ci["half_ci"].sort_values(ascending=False).index.tolist()
ypos = np.arange(len(o))[::-1]
for yy, lab in zip(ypos, o):
    h = ci.loc[lab, "half_ci"]
    c = LABEL_COLORS[lab]
    ax0.plot([TRUE_AUC - h, TRUE_AUC + h], [yy, yy], color=c, linewidth=3.0,
             solid_capstyle="round", zorder=3)
    ax0.plot([TRUE_AUC], [yy], marker="o", color=c, markersize=9,
             markeredgecolor=SURFACE, markeredgewidth=2, zorder=4)
    ax0.text(TRUE_AUC + h + 0.012, yy, f"±{h:.3f}", va="center", ha="left",
             fontsize=12, color=INK_2)
ax0.axvspan(TRUE_AUC - 0.005, TRUE_AUC + 0.005, color=INK, alpha=0.10, zorder=1)
ax0.set_yticks(ypos)
ax0.set_yticklabels(o, fontsize=12.5, color=INK)
# 1.06, not 1.02: the widest interval reaches 0.982 and its ±0.182 label has to
# sit to the right of it without running off the panel.
ax0.set_xlim(0.55, 1.06)
ax0.set_ylim(-2.4, len(o) - 0.4)
ax0.set_xlabel(f"AUC measured on {len(gold)} studies, if the true AUC were {TRUE_AUC:.2f}")
style(ax0, "x")
ax0.spines["left"].set_visible(False)
ax0.set_title("Per-label 95% intervals (Hanley-McNeil)", loc="left",
              fontsize=13.5, color=INK, pad=10)
ax0.text(TRUE_AUC + 0.014, -0.80, "the shaded band is 0.01 wide, roughly the\n"
         "margin that separates leaderboard positions",
         fontsize=11.25, color=MUTED, va="top")

ax1 = fig.add_subplot(gs[1, 0])
ax1.hist(macro, bins=44, color=PRIMARY, edgecolor=SURFACE, linewidth=0.5, zorder=3)
ymax = ax1.get_ylim()[1]
ax1.set_ylim(0, ymax * 1.30)
ax1.axvline(lo, color=ACCENT, linewidth=2.0, zorder=4)
ax1.axvline(hi, color=ACCENT, linewidth=2.0, zorder=4)
ax1.annotate("", xy=(lo, ymax * 1.09), xytext=(hi, ymax * 1.09),
             arrowprops=dict(arrowstyle="<->", color=ACCENT, linewidth=1.6))
ax1.text((lo + hi) / 2, ymax * 1.14, f"95% interval spans {hi - lo:.3f}",
         ha="center", va="bottom", fontsize=13, color=ACCENT_TEXT,
         fontweight="bold")
ax1.set_xlabel("macro AUC measured on the gold 58")
ax1.set_ylabel("bootstrap resamples")
style(ax1, "y")
ax1.set_title("Macro average, bootstrapped over studies", loc="left",
              fontsize=13.5, color=INK, pad=10)

finish(fig, f"The gold 58 resolve macro AUC only to about ±{(hi - lo) / 2:.02f}, "
            f"so it cannot rank models",
       detail=f"Per-label intervals run from ±{ci['half_ci'].min():.3f} to "
              f"±{ci['half_ci'].max():.3f}.",
       headroom=0.38,
       caption=f"Scores simulated bi-normal at a true AUC of {TRUE_AUC:.2f}, seed {SEED}; "
               f"{len(macro):,} bootstrap resamples over studies, so label co-occurrence "
               f"is preserved. The shaded band in the top panel is 0.01 wide.")

**Finding.** A 95% interval of roughly 0.08 on the macro average is the single most
important number in this notebook. Competitions of this kind are decided on differences of
0.005 to 0.02. Choosing a backbone, a slice-aggregation scheme, or a learning rate by its
score on the gold 58 is choosing by coin flip, and it will feel like signal the whole time.

Use the gold 58 for the one thing 58 studies can support: auditing the report-to-label
extraction. Per label, on these 58 studies, does the extractor agree with the radiologist?
That is a per-label agreement rate with an honest confidence interval, and it is the number
that tells you whether the 4,349 derived labels are worth training on.

Model ranking has to come from a much larger validation set built out of report-derived
labels, with folds grouped by patient. Section 6 shows why the grouping key is `PatientID`
and not `StudyInstanceUID`.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">3</span><span style="vertical-align:middle;">The reports</span></h2></div>

4,407 reports, one per study, all present. They are the only path to labels for 98.7% of
the training set, so their structure is not a curiosity, it is the input to the most
important component in the pipeline.

In [ ]:
"""Report length distribution."""

reports = train["Report"].fillna("")
lengths = reports.str.len()
words = reports.str.split().str.len()

print(f"reports present : {int(train['Report'].notna().sum()):,} of {len(train):,}")
print(f"empty strings   : {int((lengths == 0).sum()):,}")
q = lengths.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

fig, ax = plt.subplots(figsize=(14.0, 5.8))
ax.hist(lengths, bins=90, range=(0, 4000), color=PRIMARY, edgecolor=SURFACE,
        linewidth=0.4, zorder=3)
med = lengths.median()
ax.axvline(med, color=ACCENT, linewidth=2.0, zorder=4)
ax.text(med + 40, ax.get_ylim()[1] * 0.92, f"median {med:,.0f} characters",
        color=ACCENT_TEXT, fontsize=13, fontweight="bold", va="center")
p95 = lengths.quantile(0.95)
ax.axvline(p95, color=INK_2, linewidth=1.4, linestyle="-", zorder=4)
ax.text(p95 + 40, ax.get_ylim()[1] * 0.72, f"p95 {p95:,.0f}", color=INK_2,
        fontsize=12.5, va="center")
ax.set_xlabel("report length (characters)")
ax.set_ylabel("reports")
style(ax, "y")

finish(fig, "Reports are short: half are under 1,000 characters",
       detail=f"Range {lengths.min():,.0f} to {lengths.max():,.0f} characters, "
              f"median {med:,.0f}, p95 {p95:,.0f}. Median {words.median():,.0f} words.",
       caption="Histogram truncated at 4,000 characters for legibility; "
               f"{int((lengths > 4000).sum())} reports are longer.")

show_table(q.rename("characters").to_frame().assign(
    words=words.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).values).round(0))

**Finding.** These fit comfortably in any context window, so the extraction step is not
constrained by document length. A median report of about 1,000 characters costs very little
to process one at a time, and the entire corpus is a few million characters. Length is not
the obstacle. Language is.

In [ ]:
"""A language fingerprint: unambiguous script blocks, then a stopword vote."""

GREEK_RE = re.compile(r"[Ͱ-Ͽἀ-῿]")
CYRILLIC_RE = re.compile(r"[Ѐ-ӿ]")
CJK_RE = re.compile(r"[぀-ヿ一-鿿가-힯]")
ARABIC_RE = re.compile(r"[؀-ۿ]")
TURKISH_DIAC_RE = re.compile(r"[şŞğĞıİ]")
WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

STOPWORDS = {
    "English": ["the", "and", "with", "there", "normal", "signal", "joint", "tear", "no"],
    "Spanish": ["de", "la", "el", "con", "sin", "rodilla", "menisco", "los", "una"],
    "Turkish": ["ve", "ile", "yok", "diz", "normaldir", "izlenmektedir", "bulgular"],
    "German": ["der", "die", "das", "und", "mit", "kein", "keine", "nicht", "des"],
    "Dutch": ["het", "een", "van", "met", "geen", "knie", "niet", "zijn"],
    "French": ["le", "la", "les", "avec", "sans", "genou", "est", "des", "une"],
    "Italian": ["del", "della", "con", "senza", "ginocchio", "non", "menisco"],
    "Portuguese": ["do", "da", "com", "sem", "joelho", "nao", "menisco"],
}


def fingerprint(text):
    """Guess a report's language from its script, then a stopword vote.

    Script detection is exact; the stopword vote is a crude heuristic and is
    labelled as such wherever its output is shown.

    Args:
        text: Raw report text.

    Returns:
        A language or script name, or ``"Unresolved"`` when no vote wins clearly.
    """
    if GREEK_RE.search(text):
        return "Greek"
    if CYRILLIC_RE.search(text):
        return "Cyrillic script"
    if CJK_RE.search(text):
        return "CJK script"
    if ARABIC_RE.search(text):
        return "Arabic script"
    toks = collections.Counter(w.lower() for w in WORD_RE.findall(text))
    if not toks:
        return "Unresolved"
    score = {k: sum(toks[w] for w in ws) for k, ws in STOPWORDS.items()}
    if TURKISH_DIAC_RE.search(text):
        score["Turkish"] += 5
    ranked = sorted(score.items(), key=lambda kv: -kv[1])
    if ranked[0][1] < 3 or ranked[0][1] == ranked[1][1]:
        return "Unresolved"
    return ranked[0][0]


lang = reports.map(fingerprint)
lang_counts = lang.value_counts()
print(lang_counts.to_string())
print(f"\nnon-English share: {100 * (1 - lang_counts.get('English', 0) / len(lang)):.1f}%")

In [ ]:
"""Chart the language mix, emphasising the one thing that matters."""

lc = lang_counts.sort_values(ascending=False)
names = lc.index.tolist()
vals = lc.to_numpy()

# Reading the members of these two buckets shows the vote is wrong for both:
# the Italian bucket holds Spanish, the Portuguese bucket holds Croatian. Mark
# them as unreliable in the chart rather than leaving the reader to trust them.
SUSPECT = {"Italian", "Portuguese", "Unresolved"}
cols = [ACCENT if n == "English" else NEUTRAL if n in SUSPECT else PRIMARY
        for n in names]

fig, ax = plt.subplots(figsize=(14.0, 6.2))
hbar(ax, names, vals, cols,
     annot=[f"{v:,}   {100 * v / len(lang):.0f}%" for v in vals], xpad=1.30)
ax.set_xlabel("reports")
handles = [Line2D([0], [0], marker="s", linestyle="none", markersize=11, label=t,
                  markerfacecolor=c, markeredgecolor="none")
           for t, c in [("English", ACCENT), ("another identified language", PRIMARY),
                        ("vote unreliable or absent", NEUTRAL)]]
ax.legend(handles=handles, loc="lower right", labelcolor=INK_2, borderaxespad=0.8)

finish(fig, f"English is a plurality, not a majority: "
            f"{100 * (1 - lc.get('English', 0) / len(lang)):.0f}% of reports are not in English",
       detail="Greek and Cyrillic script are detected exactly. The Latin-script split "
              "comes from a stopword vote and is approximate.",
       caption="This fingerprint is deliberately crude and its bucket sizes shift with the "
               "token list. What survives any token list is the claim that matters: at "
               "least eight languages are present and no single one dominates.")

**Finding.** A label extractor tuned on English clears roughly 40% of the corpus and
silently fails on the rest. Failure will not look like an error, it will look like a
negative label, because a keyword that never matches produces a zero. That is the worst
possible failure mode: quiet, systematic, and correlated with imaging site.

The fingerprint above is itself unreliable, which is part of the finding. Its Italian and
Portuguese buckets are wrong on inspection: the Italian bucket catches Spanish text, and
the Portuguese bucket catches Croatian. Language identification on short clinical text with
heavy Latin abbreviation is not a solved sub-problem you can bolt on. Design the extraction
so it does not need to know the language.

In [ ]:
"""Verbatim report openings across languages. Truncated, never edited."""


def show_examples(picker, n_langs, width=300):
    """Print one truncated verbatim report per requested language bucket.

    Args:
        picker: Series of language labels aligned with ``reports``.
        n_langs: Ordered list of bucket names to show.
        width: Characters kept from the start of each report.
    """
    for name in n_langs:
        sub = reports[picker == name]
        if not len(sub):
            print(f"--- {name}: no report matched ---\n")
            continue
        txt = sub.iloc[0][:width].replace("\n", " ").strip()
        print(f"--- {name}  (n={len(sub):,}) " + "-" * max(0, 52 - len(name)))
        print(txt + (" ..." if len(sub.iloc[0]) > width else ""))
        print()


show_examples(lang, ["English", "Spanish", "Turkish", "Greek", "German",
                     "Dutch", "French", "Cyrillic script", "Unresolved"])

**Finding.** These are not translations of one template. The Turkish report is written as a
protocol block followed by a run of `normaldir` assertions. The Greek report opens with a
`ΤΕΧΝΙΚΗ` section and states absence with `Χωρίς`. The Dutch reports carry a Belgian
hospital header with `Klinische inlichtingen` and `Bevindingen` sections. The French one
uses a section-header style where the heading names the finding and the body says whether
it is there. Section structure, negation form, and terminology all vary, and there is no
shared skeleton to parse against.

In [ ]:
"""Worked example: why keyword extraction fails. Fracture as the test case."""

# Both Greek mu forms are needed here; see the homoglyph check at the end of
# this cell for why.
FRACTURE_TOKEN = re.compile(
    r"fractur|fraktur|fraktür|kırık|kirik|κάταγ[μµ]|καταγ[μµ]|перелом|прелом",
    re.IGNORECASE)

# Deliberately narrow. Earlier drafts of this cell included "izlenme" as a
# Turkish negator, which is wrong: "izlenmektedir" means the finding IS seen and
# "izlenmemektedir" means it is not. The loose list overstated negation.
NEGATORS = [
    "aucune", "aucun", "pas de",
    "no ", "not ", "none", "without", "absence", "negative for", "unremarkable",
    "kein", "keine",
    "geen",
    "sin ", "no se observ", "no hay",
    "izlenmemekte", "izlenmemiş", "saptanmamış", "saptanmadı", "görülmemekte",
    "yoktur", " yok", "mevcut değil",
    "χωρίς", "δεν ", "ουδεμία",
    "нет", "без", "не выявлен", "не определ",
]
PRE, POST = 40, 60   # negator search window, characters either side of the token

mentions = reports.str.contains(FRACTURE_TOKEN, regex=True)
n_mentions = int(mentions.sum())

negated = 0
for txt in reports[mentions]:
    low = txt.lower()
    for m in FRACTURE_TOKEN.finditer(low):
        window = low[max(0, m.start() - PRE):m.end() + POST]
        if any(x in window for x in NEGATORS):
            negated += 1
            break

print(f"reports containing a fracture token        : {n_mentions:,}")
print(f"of those, a negator sits within the window : {negated:,} "
      f"({100 * negated / max(n_mentions, 1):.0f}%)")
print(f"gold studies actually positive for Fracture: {int(pos['Fracture'])} of {len(gold)} "
      f"({100 * pos['Fracture'] / len(gold):.0f}%)")
print()

# Every entry below is a genuine negation, verified by reading it.
PROBES = [
    ("French", "fracture", r"Fractures?\s*:\s*\n?\s*Aucune"),
    ("English", "fracture", r"[Nn]o (acute |displaced |evidence of )?fractur"),
    ("German", "fracture", r"[Kk]eine\s+\w*[Ff]raktur"),
    ("Spanish", "fracture", r"[Ss]in\s+\w*fractur|[Nn]o\s+se\s+observan?\s+\w*fractur"),
    ("Turkish", "fracture", r"kırık[^.\n]{0,30}yok|fraktür[^.\n]{0,30}yok"),
    ("Greek", "effusion", r"[Χχ]ωρίς\s+ενδαρθρική[^.\n]{0,30}"),
    ("Bulgarian", "meniscal tear", r"без\s+данни\s+за[^.\n]{0,30}"),
    ("Dutch", "ligament tear", r"[Kk]ruisbanden[^.\n]{0,30}intact"),
]
print("Verbatim negations. A keyword extractor calls every one of these POSITIVE.\n")
for name, what, pat in PROBES:
    rx = re.compile(pat)
    hit = None
    for txt in reports:
        m = rx.search(txt)
        if m:
            s = max(0, m.start() - 30)
            hit = txt[s:m.end() + 55].replace("\n", " ⏎ ").strip()
            break
    body = f"...{hit}..." if hit else "(no match for this probe)"
    print(f"  [{name:10s} {what:14s}] {body}")

# A second trap, entirely separate from negation.
greek_reports = reports[reports.str.contains(GREEK_RE, regex=True)]
micro_sign = int(greek_reports.str.contains("µ").sum())   # MICRO SIGN
greek_mu = int(greek_reports.str.contains("μ").sum())     # GREEK SMALL MU
print(f"\nHomoglyph check across the {len(greek_reports):,} Greek reports:")
print(f"  contain MICRO SIGN U+00B5 (µ)        : {micro_sign:,} "
      f"({100 * micro_sign / max(len(greek_reports), 1):.0f}%)")
print(f"  contain GREEK SMALL LETTER MU U+03BC : {greek_mu:,}")
print("  A pattern written with the correct Greek mu misses most Greek reports.")

In [ ]:
"""Chart the gap between a token match and a true positive. One denominator."""

survives = n_mentions - negated

fig, ax = plt.subplots(figsize=(14.0, 3.9))
segments = [(negated, "a negator sits within the window", ACCENT),
            (survives, "no negator nearby", NEUTRAL)]
left = 0.0
for value, name, colour in segments:
    ax.barh([0], [value], left=left, height=0.46, color=colour, zorder=3,
            edgecolor=SURFACE, linewidth=2.4)
    ax.text(left + value / 2.0, 0, f"{value:,}\n{100 * value / n_mentions:.0f}%",
            ha="center", va="center", fontsize=15, fontweight="bold",
            color=ink_on(colour))
    left += value
ax.set_yticks([])
ax.set_ylim(-0.75, 0.55)
ax.set_xlim(0, n_mentions)
ax.set_xlabel(f"reports containing a fracture token (n = {n_mentions:,})")
style(ax, "none")
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
handles = [Line2D([0], [0], marker="s", linestyle="none", markersize=11, label=n,
                  markerfacecolor=c, markeredgecolor="none")
           for _, n, c in segments]
ax.legend(handles=handles, loc="lower left", ncol=2, labelcolor=INK_2,
          borderaxespad=0.2)

finish(fig, f"{100 * negated / max(n_mentions, 1):.0f}% of the reports that mention a "
            f"fracture negate it",
       detail=f"A keyword extractor labels all {n_mentions:,} of these reports positive. "
              f"At most {survives:,} could be.",
       caption=f"Negator list covering eight languages, searched over a -{PRE}/+{POST} "
               f"character window around each token; a lower bound on negation, not an "
               f"exact count. For scale, {int(pos['Fracture'])} of the {len(gold)} "
               f"positive-enriched gold studies are Fracture-positive.")

**Finding.** The French example is the cleanest illustration. The report reads
`Fractures :` on one line and `Aucune.` on the next. The word *fracture* is present, the
finding is absent, and the two are separated by a line break that any bag-of-words method
throws away. English has `no fracture or bone contusion`, where one negation scopes two
findings at once. Turkish puts `yok` after the noun. Greek uses `Χωρίς`. Bulgarian uses
`без данни за`.

Negation is not an edge case here, it is the majority case: about three quarters of the
reports that mention a fracture go on to negate it.

Two details are worth dwelling on, because both are the kind of thing that quietly
invalidates a result. First, an earlier draft of the cell above counted `izlenme` as a
Turkish negator. It is not. `izlenmektedir` means the finding *is* seen and
`izlenmemektedir` means it is not, so the loose pattern scored positives as negatives and
inflated the negation rate. The number above uses a narrowed list, and the fact that this
error survived a first pass is the argument, not a footnote to it.

Second, look at the homoglyph count. Most Greek reports spell mu with `µ`, the MICRO SIGN
at U+00B5, rather than `μ`, the Greek letter at U+03BC. The two are visually identical and
compare unequal. A pattern written by someone who typed Greek correctly matches almost
none of the Greek corpus, and it fails by returning zero matches, which reads as a negative
label rather than an error.

A regex list would have to encode negation scope, in every language present, with correct
handling of coordination, line breaks, and Unicode confusables. Building that is harder
than the imaging model. Extract labels with something that reads the sentence, and audit
the result against the gold 58, per label.

In [ ]:
"""Are any reports byte-identical across studies, and does that matter for folds?"""

norm = reports.str.strip().str.lower()
counts = norm.value_counts()
dupes = counts[counts > 1]

print(f"studies                      : {len(train):,}")
print(f"distinct report texts        : {norm.nunique():,}")
print(f"texts used by >1 study       : {len(dupes):,}")
print(f"studies sharing a text       : {int(dupes.sum()):,} "
      f"({dupes.sum() / len(train):.1%})")
print(f"largest single group         : {int(dupes.max()) if len(dupes) else 0} studies")

gold_mask = train[LABELS].notna().all(axis=1)
gold_dup = int(norm[gold_mask].isin(dupes.index).sum())
print(f"gold studies with a shared text: {gold_dup} of {int(gold_mask.sum())}")

top = dupes.head(6)
show_table(
    pd.DataFrame({
        "studies": top.to_numpy(),
        "language": [lang[norm == t].mode().iat[0] for t in top.index],
        "chars": [len(t) for t in top.index],
        "opening": [t[:64].replace("\n", " ") + "..." for t in top.index],
    }).reset_index(drop=True),
    caption="The largest groups are boilerplate normal reads, not copied studies.",
)

**Finding.** Report text is not unique per study. A small share of studies, a few percent,
share a byte-identical report with at least one other study, and the largest single group
runs to dozens of studies all carrying the same normal read in the same language.

These are not duplicated studies. The images differ; the words a radiologist typed happen
to be the same, which is what you would expect from a template used for an unremarkable
knee. That makes them real training data rather than something to drop.

It does change how folds must be built. Any label extractor reads one text and hands the
same twelve-value target vector to every study in the group. Split that group across folds
and the model is scored on targets it has already seen the exact source of, which flatters
the validation number without flattering the model. Keep a duplicate group whole inside one
fold, the same way patients are kept whole, and the problem disappears.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">4</span><span style="vertical-align:middle;">Reports are training-time only</span></h2></div>

The finding most likely to save a reader a week.

In [ ]:
"""Show the train and test schemas, one above the other."""

train_cols = list(train.columns)
test_cols = list(test.columns)
print("train.csv columns:", train_cols)
print("test.csv  columns:", test_cols)
print("test.csv rows    :", len(test), "(placeholder rows; the real test set is hidden)")
print()
print("columns in train.csv absent from test.csv:",
      [c for c in train_cols if c not in test_cols])

fig, axes = plt.subplots(2, 1, figsize=(12.8, 8.0),
                         gridspec_kw={"hspace": 0.34})
panels = [
    ("train.csv", axes[0],
     [("StudyInstanceUID", "present"),
      ("Report", "present"),
      ("12 label columns", "present on 58 rows")]),
    ("test.csv", axes[1],
     [("StudyInstanceUID", "present"),
      ("Report", "ABSENT"),
      ("12 label columns", "ABSENT")]),
]
for title, ax, rows_ in panels:
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.text(0.02, 0.94, title, fontsize=16, fontweight="bold", color=INK,
            family="monospace")
    y = 0.74
    for name, state in rows_:
        absent = state == "ABSENT"
        # PRIMARY_DEEP, not PRIMARY: the row label is drawn in SURFACE on top of
        # this fill, and SURFACE on PRIMARY is 4.3:1, just under the AA floor.
        ax.add_patch(plt.Rectangle((0.02, y - 0.09), 0.94, 0.17,
                                   transform=ax.transAxes,
                                   facecolor=SURFACE if absent else PRIMARY_DEEP,
                                   edgecolor=NEUTRAL if absent else "none",
                                   linewidth=1.6, linestyle=(0, (4, 3)) if absent else "-",
                                   zorder=2))
        ax.text(0.05, y - 0.005, name, transform=ax.transAxes, fontsize=15,
                color=MUTED if absent else ink_on(PRIMARY_DEEP),
                fontweight="bold" if not absent else "normal", va="center",
                family="monospace")
        ax.text(0.93, y - 0.005, state, transform=ax.transAxes, fontsize=12,
                color=ACCENT_TEXT if absent else ink_on(PRIMARY_DEEP),
                va="center", ha="right")
        y -= 0.26
    ax.text(0.02, 0.02, f"{len(train):,} rows" if title == "train.csv"
            else f"{len(test):,} placeholder rows, real test hidden",
            transform=ax.transAxes, fontsize=12, color=MUTED)

finish(fig, "test.csv has one column. There is no report at inference time",
       detail="Text is a label source and an auxiliary training signal. It is never an "
              "input to a prediction.",
       caption="Dashed outline marks a column that exists in train.csv and does not exist "
               "in test.csv. Every box is labelled, so nothing here is carried by colour alone.")

**Finding.** "Multimodal" in the competition description means the model may *learn* from
text. At inference it sees pixels and DICOM headers. Any pipeline that reads report text to
produce a prediction scores nothing, and it will look excellent in cross-validation right
up until submission, because the report contains the answer.

Three designs survive this constraint. Use text only to build labels, then train a pure
imaging model. Use text as an auxiliary target during training, for instance by distilling
a text encoder's representation into the image encoder, and drop the text branch at
inference. Or use text to build a curriculum, weighting studies by how confident the
extraction was. All three are legitimate. Reading the report at predict time is not.

There is a related trap worth naming. If you build a report-derived validation set, the
labels on it came from the report, so a model that has seen that report during training has
an advantage that will not transfer. Keep the split at the patient level and keep the
extraction step out of the fold, exactly as you would for target encoding.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">5</span><span style="vertical-align:middle;">Series structure</span></h2></div>

Five or six series per study, three planes, and two provided protocol flags.

In [ ]:
"""Series per study, plane mix, and the two protocol flags."""

spc = train_series.groupby("StudyInstanceUID").size()
planes = train_series["Anatomical_Plane"].value_counts()

print("series per study:", spc.describe()[["mean", "std", "min", "50%", "max"]].round(2).to_dict())
print("studies covered :", spc.size, "of", train["StudyInstanceUID"].nunique())
print()
print(planes.to_string())
print()
print("every study has all three planes:",
      bool((train_series.groupby("StudyInstanceUID")["Anatomical_Plane"]
            .nunique() == 3).all()))
print("plane counts per study (min across studies):",
      train_series.groupby("StudyInstanceUID")["Anatomical_Plane"].nunique().min())

fig, axes = plt.subplots(2, 1, figsize=(13.5, 9.4),
                         gridspec_kw={"height_ratios": [1.30, 1.0], "hspace": 0.34})

ax0 = axes[0]
c = spc.value_counts().sort_index()
bars = ax0.bar(c.index, c.values, width=0.72, color=PRIMARY, zorder=3)
mode_k = int(c.idxmax())
for b, k in zip(bars, c.index):
    if k == mode_k:
        b.set_color(ACCENT)
for k, v in c.items():
    if v > len(spc) * 0.01:
        ax0.text(k, v + len(spc) * 0.012, f"{v:,}", ha="center", va="bottom",
                 fontsize=11.5, color=INK_2)
ax0.set_ylim(0, c.max() * 1.12)
ax0.set_xticks(list(c.index))
ax0.set_xlabel("series in the study")
ax0.set_ylabel("studies")
style(ax0, "y")
ax0.set_title(f"Most studies have exactly {mode_k} series", loc="left",
              fontsize=13.5, color=INK, pad=10)

ax1 = axes[1]
pv = planes.to_numpy()
hbar(ax1, planes.index.tolist(), pv, [PRIMARY] * len(planes),
     annot=[f"{v:,}   {100 * v / len(train_series):.0f}%" for v in pv], xpad=1.34)
ax1.set_xlabel("series")
ax1.set_title("Sagittal is the most acquired plane", loc="left", fontsize=13.5,
              color=INK, pad=10)

finish(fig, "A study is five or six series, and all three planes are always present",
       headroom=0.38,
       detail=f"Median {int(spc.median())} series per study, range {int(spc.min())} to "
              f"{int(spc.max())}. Sagittal {planes.get('Sagittal', 0):,}, "
              f"Coronal {planes.get('Coronal', 0):,}, Axial {planes.get('Axial', 0):,}.",
       caption="Top: distribution over the 4,407 training studies. Bottom: plane mix "
               "over all 24,371 series.")

**Finding.** Series count varies from three to fourteen, so a model cannot assume a fixed
input shape at the study level. Something has to pool a variable-length set of series into
one prediction, and the obvious choices differ in how much they can learn: mean or max over
per-series logits, attention over series embeddings, or a plane-conditioned head that
consumes exactly one sagittal, one coronal and one axial series. The last one is attractive
because the plane composition is stable, and it is the reason the plane check above matters.

In [ ]:
"""Are Fluid_Sensitive and Fat_Suppression the same column? Answer, do not ask."""

fs = train_series["Fluid_Sensitive"]
fat = train_series["Fat_Suppression"]
disagree = int((fs != fat).sum())
identical = disagree == 0

print("Fluid_Sensitive value counts:", fs.value_counts().to_dict())
print("Fat_Suppression value counts:", fat.value_counts().to_dict())
print()
print(pd.crosstab(fs, fat, rownames=["Fluid_Sensitive"], colnames=["Fat_Suppression"]))
print()
print(f"rows where the two disagree : {disagree} of {len(train_series):,}")
print(f"IDENTICAL COLUMNS           : {identical}")

fs_te = test_series["Fluid_Sensitive"]
fat_te = test_series["Fat_Suppression"]
print(f"same check on test_series   : {int((fs_te != fat_te).sum())} disagreements "
      f"of {len(test_series):,} rows")

fig, ax = plt.subplots(figsize=(14.0, 5.0))
ct = pd.crosstab(train_series["Anatomical_Plane"], fs)
order = ["Sagittal", "Coronal", "Axial"]
ct = ct.reindex([p for p in order if p in ct.index])
y = np.arange(len(ct))[::-1]
left = np.zeros(len(ct))
# PRIMARY_DEEP for the segment that carries a number inside it.
seg_cols = [NEUTRAL, PRIMARY_DEEP]
seg_names = ["flag = 0", "flag = 1"]
for j, col in enumerate(ct.columns):
    w = ct[col].to_numpy(dtype=float)
    ax.barh(y, w, left=left, height=0.66, color=seg_cols[j], zorder=3,
            edgecolor=SURFACE, linewidth=2.0)
    for yy, ww, ll in zip(y, w, left):
        if ww > ct.to_numpy().sum() * 0.02:
            ax.text(ll + ww / 2, yy, f"{int(ww):,}", ha="center", va="center",
                    fontsize=12, fontweight="bold", color=ink_on(seg_cols[j]))
    left = left + w
ax.set_yticks(y)
ax.set_yticklabels(ct.index.tolist(), fontsize=12.5, color=INK)
ax.set_xlabel("series")
style(ax, "x")
ax.spines["left"].set_visible(False)
handles = [Line2D([0], [0], marker="s", linestyle="none", markersize=11, label=n,
                  markerfacecolor=c, markeredgecolor="none")
           for n, c in zip(seg_names, seg_cols)]
ax.legend(handles=handles, loc="lower right", labelcolor=INK_2, borderaxespad=0.8)

finish(fig, "Fluid_Sensitive and Fat_Suppression are the same column, byte for byte",
       detail=f"{disagree} disagreements across {len(train_series):,} training series. "
              f"Treat them as one binary protocol flag, not two features.",
       caption="Bars break the single flag down by plane. Axial series are overwhelmingly "
               "flagged 1; sagittal series are close to an even split.")

**Finding.** They are identical. Zero disagreements in 24,371 training series and zero in
the test series file. Whatever the host intended, what shipped is one binary flag under two
names, so a feature set that uses both is duplicating a column and any importance ranking
will split its credit across the pair.

The flag is not distributed evenly across planes. Axial series are flagged 1 four times as
often as flagged 0, while sagittal series split almost evenly. So the flag and the plane
carry overlapping information, and a plane-conditioned model already knows some of what the
flag would tell it. The DICOM `SeriesDescription` and `ScanningSequence` tags give an
independent read on the same protocol question, and section 6 shows they are available at
test time.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">6</span><span style="vertical-align:middle;">DICOM metadata</span></h2></div>

Headers are read with `stop_before_pixels=True` from a bounded sample of series. Nothing
here touches pixel data, so it is cheap even on the full mount.

In [ ]:
"""Discover series and read a bounded sample of headers."""

import pydicom
from pydicom.multival import MultiValue


def discover_series(max_series):
    """Find up to ``max_series`` DICOM series without walking every slice.

    Prefers the nested ``train_series/<study>/<series>/*.dcm`` layout on the
    Kaggle mount. Falls back to a bounded walk of whatever DICOM directories
    exist locally, grouping files by the SeriesInstanceUID in their header.

    Args:
        max_series: Hard cap on the number of series returned.

    Returns:
        A list of ``(study_uid, series_uid, [file paths])`` tuples, and a string
        naming the discovery mode used.
    """
    out = []
    for base in (TRAIN_DCM_DIR, TEST_DCM_DIR):
        if not base.is_dir():
            continue
        studies = sorted(p.name for p in base.iterdir() if p.is_dir())
        if not studies:
            continue
        k = min(N_HEADER_STUDIES, len(studies))
        pick = np.random.default_rng(SEED).choice(len(studies), size=k, replace=False)
        for i in sorted(pick):
            sdir = base / studies[i]
            for ser in sorted(p for p in sdir.iterdir() if p.is_dir()):
                files = sorted(str(f) for f in ser.glob("*.dcm"))
                if files:
                    out.append((studies[i], ser.name, files))
                if len(out) >= max_series:
                    return out, f"nested layout under {base.name}"
        if out:
            return out, f"nested layout under {base.name}"

    # Fallback: a flat or unknown local layout. Bounded, then grouped by header.
    files = []
    for d in [TRAIN_DCM_DIR, TEST_DCM_DIR, *EXTRA_DCM_DIRS]:
        if not d.is_dir():
            continue
        for dirpath, _, filenames in os.walk(d):
            for fn in sorted(filenames):
                if fn.lower().endswith(".dcm"):
                    files.append(os.path.join(dirpath, fn))
                    if len(files) >= MAX_FALLBACK_FILES:
                        break
            if len(files) >= MAX_FALLBACK_FILES:
                break
        if len(files) >= MAX_FALLBACK_FILES:
            break
    groups = collections.defaultdict(list)
    for f in sorted(files):
        try:
            d = pydicom.dcmread(f, stop_before_pixels=True, specific_tags=[
                "StudyInstanceUID", "SeriesInstanceUID"])
            groups[(str(d.StudyInstanceUID), str(d.SeriesInstanceUID))].append(f)
        except Exception:
            continue
    out = [(a, b, sorted(v)) for (a, b), v in sorted(groups.items())][:max_series]
    return out, "bounded walk, grouped by SeriesInstanceUID"


series_index, discovery_mode = discover_series(MAX_HEADER_SERIES)
n_files_seen = sum(len(f) for _, _, f in series_index)
print(f"discovery mode : {discovery_mode}")
print(f"series found   : {len(series_index):,}")
print(f"slices indexed : {n_files_seen:,}")
if not series_index:
    print("\nNO DICOM FILES FOUND. Sections 6 and 7 will report that and continue.")

In [ ]:
"""Read one header per discovered series into a dataframe."""

HEADER_TAGS = [
    "PatientID", "PatientSex", "Laterality", "BodyPartExamined",
    "Manufacturer", "ManufacturerModelName", "MagneticFieldStrength",
    "SoftwareVersions", "ReceiveCoilName", "SeriesDescription",
    "ScanningSequence", "SequenceVariant", "MRAcquisitionType",
    "RepetitionTime", "EchoTime", "InversionTime", "FlipAngle",
    "EchoTrainLength", "SliceThickness", "SpacingBetweenSlices",
    "PixelSpacing", "Rows", "Columns", "BitsStored", "PhotometricInterpretation",
    "WindowCenter", "WindowWidth", "RescaleSlope", "RescaleIntercept",
    "InstanceNumber", "SeriesNumber", "ImageOrientationPatient",
]


def plane_from_iop(iop):
    """Derive the acquisition plane from ImageOrientationPatient.

    The slice normal is the cross product of the two row/column direction
    cosines; whichever patient axis it aligns with names the plane.

    Args:
        iop: Six-element ImageOrientationPatient value, or ``None``.

    Returns:
        ``"Sagittal"``, ``"Coronal"``, ``"Axial"`` or ``"unknown"``.
    """
    try:
        v = np.asarray([float(x) for x in iop], dtype=float)
        n = np.abs(np.cross(v[:3], v[3:]))
        return ["Sagittal", "Coronal", "Axial"][int(np.argmax(n))]
    except Exception:
        return "unknown"


rows = []
header_failures = []
for study_uid, series_uid, files in series_index:
    try:
        ds = pydicom.dcmread(files[len(files) // 2], stop_before_pixels=True)
    except Exception as exc:
        header_failures.append((series_uid, type(exc).__name__, str(exc)[:90]))
        continue
    rec = {"StudyInstanceUID": study_uid, "SeriesInstanceUID": series_uid,
           "n_slices": len(files), "path": files[len(files) // 2]}
    for tag in HEADER_TAGS:
        val = getattr(ds, tag, None)
        if isinstance(val, MultiValue):
            val = list(val)
        rec[tag] = val
    try:
        rec["TransferSyntax"] = str(ds.file_meta.TransferSyntaxUID.name)
    except Exception:
        rec["TransferSyntax"] = "unknown"
    rec["plane_from_iop"] = plane_from_iop(rec.get("ImageOrientationPatient"))
    rows.append(rec)

hdr = pd.DataFrame(rows)
print(f"headers read    : {len(hdr):,}")
print(f"header failures : {len(header_failures)}")
for f in header_failures[:5]:
    print("   ", f)

if len(hdr):
    known = pd.concat([train_series, test_series], ignore_index=True)[
        ["SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive"]]
    hdr = hdr.merge(known, on="SeriesInstanceUID", how="left")
    matched = hdr["Anatomical_Plane"].notna() & (hdr["plane_from_iop"] != "unknown")
    if matched.any():
        agree = (hdr.loc[matched, "Anatomical_Plane"]
                 == hdr.loc[matched, "plane_from_iop"]).mean()
        print(f"\nplane derived from ImageOrientationPatient agrees with the provided "
              f"Anatomical_Plane on {100 * agree:.1f}% of {int(matched.sum()):,} "
              f"matched series")
        print("(the derived plane comes from the direction cosines in the slice's own "
              "header, so it is an independent check on the CSV rather than a copy of it)")
    display_cols = ["Manufacturer", "ManufacturerModelName", "MagneticFieldStrength",
                    "SeriesDescription", "Laterality", "PatientSex", "Rows",
                    "SliceThickness", "TransferSyntax", "n_slices"]
    show_table(hdr[[c for c in display_cols if c in hdr.columns]].head(8))

In [ ]:
"""Scanner, protocol and encoding census over the sampled series."""

if not len(hdr):
    print("No DICOM headers available at this data root; skipping the census charts.")
else:
    def top_counts(col, k=8):
        """Return the top ``k`` value counts of a header column as strings."""
        s = hdr[col].dropna().astype(str)
        s = s[s.str.strip() != ""]
        return s.value_counts().head(k)

    # State what the sample actually contains rather than asserting variety the
    # panels may not show: on a small local mirror every facet has one value.
    def n_distinct(col):
        """Count distinct non-empty values of a header column."""
        if col not in hdr.columns:
            return 0
        s = hdr[col].dropna().astype(str)
        return int(s[s.str.strip() != ""].nunique())

    def census_pair(panels, finding, caption_tail):
        """Draw two stacked category panels and close the figure out.

        Args:
            panels: Two ``(column, title)`` pairs, drawn top then bottom.
            finding: The headline for this figure.
            caption_tail: Sentence appended to the shared provenance caption.
        """
        fig, axes = plt.subplots(2, 1, figsize=(13.5, 9.8),
                                 gridspec_kw={"hspace": 0.40})
        for ax, (col, title) in zip(axes, panels):
            if col not in hdr.columns:
                ax.axis("off")
                continue
            vc = top_counts(col, 8)
            if not len(vc):
                ax.axis("off")
                ax.text(0.5, 0.5, f"{col}: no values", ha="center", va="center",
                        color=MUTED, transform=ax.transAxes)
                continue
            names = [n if len(n) <= 34 else n[:32] + "…" for n in vc.index.tolist()]
            cols_ = [ACCENT if i == 0 else PRIMARY for i in range(len(vc))]
            hbar(ax, names, vc.to_numpy(), cols_, fmt="{:,.0f}")
            # Reserve four bar slots even when the sample holds fewer values,
            # padding downward so the bars stay top-aligned. On the full mount
            # every facet fills all eight and this does nothing; on a one-vendor
            # local mirror it stops a single bar swelling to half the panel.
            ax.set_ylim(len(vc) - max(len(vc), 4) - 0.75, len(vc) - 0.25)
            ax.set_title(title, loc="left", fontsize=13.5, color=INK, pad=10)
            ax.set_xlabel("series sampled")
        finish(fig, finding, headroom=0.38,
               detail=f"Census over {len(hdr):,} series sampled from "
                      f"{hdr['StudyInstanceUID'].nunique():,} studies, one header "
                      f"each. A small local mirror will show one value per facet; "
                      f"the full 265 GB mount does not.",
               caption=f"Bounded by N_HEADER_STUDIES={N_HEADER_STUDIES} and "
                       f"MAX_HEADER_SERIES={MAX_HEADER_SERIES}, seed {SEED}. "
                       f"Discovery mode: {discovery_mode}. Top 8 values per panel. "
                       + caption_tail)

    n_vendor = n_distinct("Manufacturer")
    n_field = n_distinct("MagneticFieldStrength")
    n_syntax = n_distinct("TransferSyntax")
    n_model = n_distinct("ManufacturerModelName")

    census_pair(
        [("Manufacturer", "Scanner vendor"),
         ("ManufacturerModelName", "Scanner model")],
        f"{n_vendor} vendor{'' if n_vendor == 1 else 's'} and {n_model} scanner "
        f"model{'' if n_model == 1 else 's'} in this sample",
        "Vendor and model are the closest thing the corpus has to a site label.")

    census_pair(
        [("MagneticFieldStrength", "Field strength (T)"),
         ("TransferSyntax", "Pixel encoding")],
        f"{n_field} field strength{'' if n_field == 1 else 's'} and {n_syntax} "
        f"pixel encoding{'' if n_syntax == 1 else 's'} in this sample",
        "Pixel encoding decides which decoders a submission kernel has to ship.")

**Finding.** The host describes "a diverse international mix of imaging sites". Vendor,
model, field strength, coil and software version are all recorded in the header, which
makes them a usable proxy for the site. That cuts two ways. They are the right variables
for a domain-shift check, and they are exactly the shortcut a network will find if one site
happens to contribute most of the positives for a finding. Check whether any label's
prevalence correlates with vendor before trusting a validation score.

Pixel encoding matters for a more practical reason. Transfer syntaxes are mixed across the
full corpus, so the decode path has to handle uncompressed Explicit VR, Implicit VR, JPEG
Lossless and JPEG 2000. A submission kernel that only decodes uncompressed data does not
score badly on the hidden test set, it fails. Verify the decoders in the submission
environment rather than trusting whatever the interactive session happened to sample.
Kaggle's image includes the JPEG plugins; a self-built container may not.

In [ ]:
"""Patient grouping, laterality and sex. The fold-construction question."""

if not len(hdr):
    print("No DICOM headers available; skipping the patient-grouping check.")
else:
    pid = hdr.dropna(subset=["PatientID"])
    per_patient = pid.groupby("PatientID")["StudyInstanceUID"].nunique()
    multi = int((per_patient > 1).sum())
    print(f"distinct PatientIDs in the sample : {per_patient.size:,}")
    print(f"studies in the sample             : {pid['StudyInstanceUID'].nunique():,}")
    print(f"patients contributing >1 study    : {multi}")
    print(f"studies per patient (max)         : {int(per_patient.max())}")
    print()
    print("PatientID example (pseudonymised):", pid["PatientID"].iloc[0])
    print()
    print("Laterality:", hdr["Laterality"].dropna().astype(str).value_counts().to_dict())
    print("PatientSex:", hdr["PatientSex"].dropna().astype(str).value_counts().to_dict())
    print()
    per_study_pid = pid.groupby("StudyInstanceUID")["PatientID"].nunique()
    print("studies whose series disagree on PatientID:",
          int((per_study_pid > 1).sum()), "of", per_study_pid.size)

    fig, axes = plt.subplots(2, 1, figsize=(13.0, 6.6),
                             gridspec_kw={"hspace": 0.50})
    lat = hdr["Laterality"].dropna().astype(str).value_counts()
    sex = hdr["PatientSex"].dropna().astype(str).value_counts()
    for ax, vc, title, xl in [(axes[0], lat, "Which knee", "series"),
                              (axes[1], sex, "Patient sex", "series")]:
        if len(vc):
            hbar(ax, vc.index.tolist(), vc.to_numpy(), [PRIMARY] * len(vc))
            ax.set_xlabel(xl)
        else:
            ax.axis("off")
            ax.text(0.5, 0.5, "tag absent", ha="center", va="center", color=MUTED,
                    transform=ax.transAxes)
        ax.set_title(title, loc="left", fontsize=13.5, color=INK, pad=10)

    finish(fig, "Laterality and patient sex are in the DICOM header, and so is a patient key",
           headroom=0.38, footroom=0.16,
           detail="PatientID is pseudonymised but stable, which makes it the grouping key "
                  "for cross-validation.",
           caption=f"Counted over the {len(hdr):,} sampled series. "
                   "Laterality matters: a left-knee model must not learn the mirror image.")

**Finding.** `PatientID` is present and pseudonymised; the cell above prints one, along
with the number of patients contributing more than one study in the sampled slice of the
corpus. Group folds by `PatientID`, not by `StudyInstanceUID`. A patient with two studies
puts the same knee on both sides of a study-level split, and knee MRI is about as
patient-specific as imaging gets: the same degenerative changes appear in both. Grouping
costs nothing when every patient turns out to have exactly one study, so build the
study-to-patient map over the whole corpus and group on it either way.

`Laterality` deserves attention for a different reason. Left and right knees are mirror
images, so either normalise by flipping all rights to lefts, or feed laterality to the model
explicitly. Doing neither forces the network to spend capacity learning a symmetry it was
handed for free. Both `Laterality` and `PatientSex` come from the DICOM header rather than
`train.csv`, so both are available at test time.

In [ ]:
"""What the SeriesDescription strings say about the protocol."""

if not len(hdr) or "SeriesDescription" not in hdr.columns:
    print("SeriesDescription unavailable; skipping.")
else:
    desc = hdr["SeriesDescription"].dropna().astype(str).str.lower()
    print(f"distinct SeriesDescription values in the sample: {desc.nunique():,} "
          f"over {len(desc):,} series")
    print("\nmost common raw values:")
    print(desc.value_counts().head(12).to_string())

    tokens = collections.Counter()
    for d in desc:
        for t in re.split(r"[^a-z0-9]+", d):
            if t and not t.isdigit() and len(t) > 1:
                tokens[t] += 1
    top = tokens.most_common(18)

    fig, ax = plt.subplots(figsize=(14.0, 7.4))
    PROTOCOL_HINTS = {"fs", "stir", "spair", "sag", "cor", "tra", "ax", "t1", "t2",
                      "pd", "dp", "tirm", "fatsat", "sagittal", "coronal", "axial"}
    names = [t for t, _ in top]
    vals = [c for _, c in top]
    cols_ = [ACCENT if n in PROTOCOL_HINTS else PRIMARY for n in names]
    hbar(ax, names, vals, cols_, fmt="{:,.0f}")
    ax.set_xlabel("series containing the token")
    handles = [Line2D([0], [0], marker="s", linestyle="none", markersize=11, label=t,
                      markerfacecolor=c, markeredgecolor="none")
               for t, c in [("plane or contrast token", ACCENT), ("other token", PRIMARY)]]
    ax.legend(handles=handles, loc="lower right", labelcolor=INK_2, borderaxespad=0.8)

    finish(fig, "SeriesDescription encodes plane and contrast weighting in free text",
           detail="Tokens like sag, cor, tra, pd, t2 and fs recover the acquisition "
                  "protocol independently of the provided CSV flags.",
           caption=f"Tokenised on non-alphanumeric boundaries over {len(desc):,} sampled "
                   "series; pure digits dropped. Vendor naming is inconsistent, so treat "
                   "this as a signal to verify, not a parser to trust.")

**Finding.** `SeriesDescription` is vendor free text and not a controlled vocabulary, so
parsing it is a heuristic. It is still useful as a cross-check on `Anatomical_Plane` and on
the single protocol flag from section 5, and it is available at test time. The stronger
independent check is geometric: the plane derived from `ImageOrientationPatient` is
computed from direction cosines rather than from a string, and the agreement rate against
the provided `Anatomical_Plane` is printed above.

**Which tags are usable at inference.** All of them. The test DICOMs come from the same
de-identification pipeline as the training DICOMs, so `PatientID`, `Laterality`,
`PatientSex`, `Manufacturer`, `MagneticFieldStrength`, `SeriesDescription`, the geometry
tags and the windowing tags are present at test time exactly as they are here. The only
thing missing at test time is the report. That asymmetry is worth stating in reverse: DICOM
metadata is fair game as model input, free text is not.

One caveat on `PatientID` at test time. It is legitimate as a *feature* only if you are
comfortable with what it implies, and it is far more valuable as a grouping key for
test-time aggregation: if two test studies share a patient, their predictions are not
independent.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">7</span><span style="vertical-align:middle;">Pixels</span></h2></div>

A montage across the three planes with correct windowing, then the reason per-series
normalisation is not optional.

In [ ]:
"""Windowing helpers, then a montage across the three acquisition planes."""


def load_windowed(path):
    """Read one slice and map it to display range with the DICOM's own window.

    Applies RescaleSlope and RescaleIntercept, then WindowCenter and WindowWidth
    when the header carries them, falling back to the 1st-99th percentile.
    MONOCHROME1 is inverted so bright always means high signal.

    Args:
        path: Path to a DICOM file.

    Returns:
        A ``(H, W)`` float array in ``[0, 1]``, and the dataset, or
        ``(None, None)`` if the pixel data could not be decoded.
    """
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None, None
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    inter = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + inter
    wc, ww = getattr(ds, "WindowCenter", None), getattr(ds, "WindowWidth", None)
    try:
        wc = float(wc[0] if isinstance(wc, MultiValue) else wc)
        ww = float(ww[0] if isinstance(ww, MultiValue) else ww)
    except (TypeError, ValueError):
        wc = ww = None
    if wc is None or ww is None or ww <= 0:
        lo_, hi_ = np.percentile(arr, [1, 99])
    else:
        lo_, hi_ = wc - ww / 2.0, wc + ww / 2.0
    if hi_ <= lo_:
        lo_, hi_ = float(arr.min()), float(arr.max()) or 1.0
    out = np.clip((arr - lo_) / (hi_ - lo_), 0.0, 1.0)
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")) == "MONOCHROME1":
        out = 1.0 - out
    return out, ds


PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
pixel_failures = []

if not len(hdr):
    print("No DICOM data at this root; skipping the montage.")
    montage = {}
else:
    files_by_series = {s: f for _, s, f in series_index}
    # Prefer the plane computed from the slice's own direction cosines: the
    # panel label then describes the pixels actually on screen, whatever the
    # CSV says.
    plane_col = hdr["plane_from_iop"].where(hdr["plane_from_iop"] != "unknown",
                                            hdr["Anatomical_Plane"])
    montage = {}
    for plane in PLANE_ORDER:
        cand = hdr.index[plane_col == plane].tolist()
        if not cand:
            continue
        cand = cand[:N_MONTAGE_CANDIDATES]
        for i in cand:
            row = hdr.loc[i]
            files = files_by_series.get(row["SeriesInstanceUID"], [])
            if len(files) < 2:
                continue
            k = min(N_MONTAGE_SLICES, len(files))
            idx = (np.linspace(len(files) * 0.25, len(files) * 0.75, k)
                   .round().astype(int).clip(0, len(files) - 1))
            # Short series round two positions onto the same slice, which would
            # put the identical image in two panels. Keep first occurrences.
            seen, keep = set(), []
            for j in idx.tolist():
                if j not in seen:
                    seen.add(j)
                    keep.append(j)
            picks = [files[j] for j in keep]
            imgs = []
            for p in picks:
                img, _ = load_windowed(p)
                if img is None:
                    pixel_failures.append(p)
                else:
                    imgs.append(img)
            if imgs:
                montage[plane] = (row, imgs)
                break
    print("planes with decodable pixel data:", list(montage))
    print("slice decode failures:", len(pixel_failures))

In [ ]:
"""Render the montage: three planes down, slices across, one greyscale."""

if not montage:
    print("No decodable pixel data available at this data root. "
          "On the Kaggle mount this cell renders a 3 x "
          f"{N_MONTAGE_SLICES} montage.")
else:
    ncol = max(len(v[1]) for v in montage.values())
    nrow = len(montage)
    # The montage is the visual centrepiece of the notebook, so it is the one
    # grid that gets the full column width. Width is capped at 14.4 inches so a
    # five-slice run cannot push it past what Kaggle's column can show.
    cell_w = min(3.45, 14.4 / max(ncol, 1))
    fig, axes = plt.subplots(nrow, ncol, figsize=(cell_w * ncol, 3.65 * nrow),
                             squeeze=False)
    stroke = [pe.withStroke(linewidth=2.8, foreground="#000000")]
    for r, plane in enumerate([p for p in PLANE_ORDER if p in montage]):
        row, imgs = montage[plane]
        desc = str(row.get("SeriesDescription") or "series description absent")
        for c in range(ncol):
            ax = axes[r][c]
            ax.set_xticks([])
            ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            if c >= len(imgs):
                ax.axis("off")
                continue
            ax.imshow(imgs[c], cmap="gray", vmin=0.0, vmax=1.0,
                      interpolation="bilinear")
            if c == 0:
                ax.text(0.035, 0.960, plane, transform=ax.transAxes, color="#ffffff",
                        fontsize=15, fontweight="bold", va="top", ha="left",
                        path_effects=stroke)
                ax.text(0.035, 0.885, desc[:26], transform=ax.transAxes,
                        color="#ffffff", fontsize=11.5, va="top", ha="left",
                        path_effects=stroke)
            ax.text(0.965, 0.035, f"slice {c + 1}", transform=ax.transAxes,
                    color="#ffffff", fontsize=11, va="bottom", ha="right",
                    path_effects=stroke)
    top_in = 1.18
    bot_in = 0.50
    h_in = fig.get_size_inches()[1]
    fig.subplots_adjust(left=0.004, right=0.996, wspace=0.010, hspace=0.014,
                        top=1.0 - top_in / h_in, bottom=bot_in / h_in)

    det = (f"One series per plane, {ncol} slices each, drawn from the middle half of "
           f"the stack. Windowed with the header's own WindowCenter/WindowWidth.")
    finish(fig, "Three planes, three different views of the same joint", detail=det,
           tight=False,
           caption=f"RescaleSlope/Intercept applied before windowing; MONOCHROME1 "
                   f"inverted. {len(pixel_failures)} slice decode failures in this sample. "
                   f"Sequences shown: "
                   + "; ".join(f"{p}: {str(montage[p][0].get('SeriesDescription'))}"
                               for p in montage))

**Finding.** The three planes are not redundant views of one thing, they are the reason a
radiologist reads a knee in three planes. Cruciate ligaments run obliquely and are read
sagittally. Collateral ligaments and the meniscal body are read coronally. Patellar
cartilage and the retinacula are read axially. Since the twelve findings map unevenly onto
the planes, a model that pools all series identically is throwing away the structure the
protocol was designed around. Condition on the plane.

Windowing is not cosmetic. Applying `RescaleSlope` and `RescaleIntercept` before the window
is required, and skipping the window entirely leaves most of the dynamic range unused on
these images.

In [ ]:
"""Do intensities compare across series? Small multiples, same axis."""

if not len(hdr):
    print("No DICOM data; skipping the intensity comparison.")
else:
    files_by_series = {s: f for _, s, f in series_index}
    picked, stats = [], []
    for i in hdr.index:
        if len(picked) >= N_INTENSITY_SERIES:
            break
        row = hdr.loc[i]
        files = files_by_series.get(row["SeriesInstanceUID"], [])
        if not files:
            continue
        try:
            ds = pydicom.dcmread(files[len(files) // 2])
            arr = ds.pixel_array.astype(np.float32)
            arr = arr * float(getattr(ds, "RescaleSlope", 1.0) or 1.0) \
                + float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        except Exception as exc:
            pixel_failures.append((files[0], type(exc).__name__))
            continue
        picked.append((row, arr))
        p1, p50, p99 = np.percentile(arr, [1, 50, 99])
        stats.append({
            "SeriesDescription": str(row.get("SeriesDescription"))[:24],
            "plane": str(row.get("plane_from_iop") or row.get("Anatomical_Plane")),
            "p1": round(float(p1), 1), "median": round(float(p50), 1),
            "p99": round(float(p99), 1), "max": round(float(arr.max()), 1),
        })

    if not picked:
        print("No decodable series for the intensity comparison.")
    else:
        hi_all = max(np.percentile(a, 99.5) for _, a in picked)
        n = len(picked)
        ncol = min(3, n)
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.7 * nrow),
                                 squeeze=False, sharex=True)
        for k, (row, arr) in enumerate(picked):
            ax = axes[k // ncol][k % ncol]
            ax.hist(arr.ravel(), bins=90, range=(0, hi_all), color=PRIMARY,
                    edgecolor="none", zorder=3, log=True)
            # Headroom above the tallest bin, so the median label sits in clear
            # space. Without it a series whose median lands on its own peak
            # writes the label straight over the bars.
            ax.set_ylim(top=ax.get_ylim()[1] * 8.0)
            med = float(np.median(arr))
            ax.axvline(med, color=ACCENT, linewidth=1.8, zorder=4)
            ax.text(med + hi_all * 0.02, ax.get_ylim()[1] * 0.30,
                    f"median {med:,.0f}", color=ACCENT_TEXT, fontsize=11,
                    va="center")
            ttl = str(row.get("SeriesDescription") or "series")[:24]
            ax.set_title(f"{ttl}  ·  {row.get('plane_from_iop') or row.get('Anatomical_Plane')}",
                         loc="left", fontsize=12.5, color=INK)
            style(ax, "y")
            ax.set_yticklabels([])
        for k in range(n, nrow * ncol):
            axes[k // ncol][k % ncol].axis("off")
        for c in range(ncol):
            axes[nrow - 1][c].set_xlabel("pixel value after rescale")

        meds = [s["median"] for s in stats]
        n_studies_shown = len({r["StudyInstanceUID"] for r, _ in picked})
        finish(fig, "Raw intensity is not comparable across series",
               headroom=0.40,
               detail=f"Medians run from {min(meds):,.0f} to {max(meds):,.0f} across "
                      f"these {n} series from {n_studies_shown} "
                      f"stud{'y' if n_studies_shown == 1 else 'ies'}. MR signal has no "
                      f"absolute unit.",
               caption=f"Log-scaled counts, shared x-axis to {hi_all:,.0f}. One mid-stack "
                       f"slice per series. Compare with CT, where Hounsfield units are "
                       f"absolute and a fixed window is meaningful everywhere.")

        show_table(pd.DataFrame(stats),
                   caption="Per-series percentiles after RescaleSlope/Intercept.")

**Finding.** MR intensity is arbitrary. There is no Hounsfield-unit equivalent, so the same
tissue takes a different number on a different sequence, a different coil, or a different
day. A fixed global normalisation applied across this corpus will hand the network a
brightness offset that tracks the acquisition site, which is a shortcut correlated with
whichever findings that site over-represents.

Normalise per series. The simple and hard-to-beat choice scales to the 1st and 99th
percentile of the series, computed over the whole volume rather than per slice, so
slices keep their relative contrast. Z-scoring on the non-background voxels is the other
reasonable option. Whichever you pick, compute it inside the fold and apply the identical
transform at test time.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;text-align:center;background:#256abf;color:#ffffff;border-radius:6px;font-size:14px;font-weight:700;margin-right:12px;vertical-align:middle;">8</span><span style="vertical-align:middle;">What this means for modelling</span></h2></div>

Nine decisions that follow from what is above, and one thing worth measuring before
committing to any of them.

**Build the label extractor first, and treat it as the main model.** 4,349 reports have to
become 52,188 binary decisions. Everything downstream is capped by how good those decisions
are. A keyword or regex approach has to survive seven or more languages and a negation rate
around three quarters on at least one finding; that is a harder engineering problem than
the imaging model, and it fails silently when it fails. Use something that reads the
sentence.

**Audit the extractor on the gold 58, per label, and report agreement with an interval.**
That is the one job 58 studies can do. Twelve agreement rates with honest intervals tell you
which findings the extraction is reliable on and which need a different prompt or a
different rule. Do not compute a macro AUC on those 58 and use it to pick anything.

**Rank models on a report-derived validation set, not the gold set.** Hold out several
hundred studies, label them with the audited extractor, and rank there. Group folds by
`PatientID`. A patient with two studies would otherwise land on both sides of the split,
and knee anatomy is patient-specific. Build the study-to-patient map over the full corpus
before splitting.

**Never read report text at inference.** Text is a label source, an auxiliary training
target, or a curriculum weight. It is not an input. A validation score that uses it will
look excellent and transfer to nothing.

**Condition on the plane.** All three planes are present in every study, the composition is
stable, and different findings live in different planes. A plane-conditioned head, or at
minimum a plane embedding, uses structure the protocol already provides.

**Handle a variable number of series.** Three to fourteen, median five. Aggregation across
series is a design decision that has to be made explicitly, and attention over series
embeddings is the version that can learn which series matters for which finding.

**Normalise per series, inside the fold.** Percentile scaling over the whole volume.
Global normalisation encodes the acquisition site.

**Use `Fluid_Sensitive` once.** It and `Fat_Suppression` are the same column, with zero
disagreements in 24,371 rows. Including both duplicates a feature and splits its importance.

**Spend on the rare findings.** Macro averaging means MCL, with nine positives in the gold
set, is worth exactly what Effusion is worth. The temptation is to optimise the average by
improving what is already good. Under a macro metric that is close to worthless.

**And measure this first:** whether label prevalence correlates with scanner vendor or
field strength. The header census shows the site proxies are all present and varied. If one
site supplies most of the positives for a finding, a network will learn the site, and both
your validation and the public leaderboard will reward it right up until the private split.

<div style="background:#fcfcfb;border:1px solid #e1e0d9;border-left:6px solid #eb6834;border-radius:7px;padding:14px 18px;margin:26px 0 14px 0;"><h2 style="color:#0b0b0b;font-size:21px;margin:0;font-weight:700;letter-spacing:-.005em;"><span style="vertical-align:middle;">How this notebook was made</span></h2></div>

This notebook was generated by an agentic system rather than typed by hand. The system is
Claude Code running a multi-agent workflow: parallel agents wrote the analysis sections
while separate adversarial verifier agents executed every cell against a local mirror of
the Kaggle input mount, checked every printed number against a fact sheet built from the
Kaggle API, opened every figure and looked at it, then sent each defect back to the builder
script instead of patching the notebook. The verifier agents are the part that matters,
because a notebook that has never been run is a draft.

The source of truth is a Python builder script, `notebooks/build_eda_notebook.py`, which
constructs the `.ipynb` with `nbformat`. The notebook is an artifact of that script. Review
happens on plain Python rather than on notebook JSON, and the notebook is regenerated
rather than edited. Publication uses the Kaggle CLI, `kaggle kernels push`, and the
companion baseline notebook is submitted to the competition the same way.

The competition facts this notebook is built on were verified from the Kaggle API and from
the data itself on 2026-08-05, the day the competition opened. Study and series counts,
the 58 labelled studies, the twelve label names and their submission order, the
identity of `Fluid_Sensitive` and `Fat_Suppression`, and the single-column `test.csv` are
all recomputed at run time in the cells above rather than quoted from that check. The
system has produced no score on this competition and this notebook claims none.

In [ ]:
"""Environment and reproducibility footer."""

import platform

print("python      ", platform.python_version())
for mod in ["numpy", "pandas", "matplotlib", "scipy", "pydicom"]:
    try:
        print(f"{mod:12s}", __import__(mod).__version__)
    except Exception as exc:
        print(f"{mod:12s} unavailable ({type(exc).__name__})")
print()
print("data root         ", ROOT)
print("seed              ", SEED)
print("header budget     ", f"{N_HEADER_STUDIES} studies / {MAX_HEADER_SERIES} series")
print("series indexed    ", len(series_index))
print("headers read      ", len(hdr))
print("header failures   ", len(header_failures))
print("pixel decode fails", len(pixel_failures))